# Semantic Index Notebook

Shows how to build and search a FAISS-based semantic index.

In [ ]:
from __future__ import annotations
from typing import List, Optional, Sequence, Tuple
import numpy as np

In [ ]:
class FAISSIndexManager:
    """Production-facing index: build over (embeddings, docs) pairs, search
    returns the *documents* (not raw vectors/ids)."""

    def __init__(self, metric: str = "cosine"):
        self.metric = metric
        self.index = None
        self.docs: List[str] = []
        self.dim: Optional[int] = None
        self._use_faiss = False

    def build_index(self, embeddings: np.ndarray, docs: Sequence[str]) -> None:
        embeddings = np.ascontiguousarray(embeddings, dtype="float32")
        assert embeddings.shape[0] == len(docs), "embeddings/docs length mismatch"
        self.docs = list(docs)
        self.dim = embeddings.shape[1]

        try:
            import faiss
            index = faiss.IndexFlatIP(self.dim) if self.metric == "cosine" else faiss.IndexFlatL2(self.dim)
            index.add(embeddings)
            self.index = index
            self._use_faiss = True
        except ImportError:
            # Brute-force numpy fallback — keeps the same behavior, just slower.
            self.index = embeddings
            self._use_faiss = False

    def search(self, query_embedding: np.ndarray, top_k: int = 3) -> List[str]:
        if self.index is None:
            return []
        query = np.ascontiguousarray(query_embedding, dtype="float32").reshape(1, -1)

        if self._use_faiss:
            _, indices = self.index.search(query, min(top_k, len(self.docs)))
            idxs = [i for i in indices[0] if i != -1]
        else:
            if self.metric == "cosine":
                scores = self.index @ query[0]
            else:
                scores = -np.linalg.norm(self.index - query[0], axis=1)
            idxs = np.argsort(-scores)[:top_k]

        return [self.docs[i] for i in idxs]

In [ ]:
class SemanticIndex:
    """Lightweight demo/teaching variant: build over raw embeddings only
    (no doc payload required) and return (distances, neighbor_indices)."""

    def __init__(self, embedding_dim: int, metric: str = "l2"):
        self.embedding_dim = embedding_dim
        self.metric = metric
        self._vectors: Optional[np.ndarray] = None

    def build(self, embeddings: np.ndarray) -> None:
        self._vectors = np.ascontiguousarray(embeddings, dtype="float32")

    def search(self, query: np.ndarray, top_k: int = 2) -> Tuple[np.ndarray, np.ndarray]:
        if self._vectors is None:
            raise RuntimeError("Call .build(embeddings) before .search().")
        query = np.asarray(query, dtype="float32")

        if self.metric == "cosine":
            sims = self._vectors @ query
            order = np.argsort(-sims)[:top_k]
            return sims[order], order
        else:  # "l2"
            dists = np.linalg.norm(self._vectors - query, axis=1)
            order = np.argsort(dists)[:top_k]
            return dists[order], order